<a href="https://colab.research.google.com/github/MeenakshiRajpurohit/RAG-Retrieval-Augmented-Generation-/blob/main/FT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
"""
Spectral RAG vs Standard RAG — Side-by-Side Comparison
=======================================================
Runs BOTH pipelines on identical data and evaluates:
  - Accuracy
  - Loss curve
  - Convergence speed
  - Embedding change magnitude
  - Parameter count
"""

import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
from typing import List, Tuple, Dict
from copy import deepcopy

torch.manual_seed(42)

# ─────────────────────────────────────────────────────────────────────────────
# SHARED COMPONENTS
# ─────────────────────────────────────────────────────────────────────────────

class ToyEmbeddingModel(nn.Module):
    def __init__(self, embed_dim: int = 64, vocab_size: int = 128):
        super().__init__()
        self.emb  = nn.Embedding(vocab_size, embed_dim)
        self.proj = nn.Linear(embed_dim, embed_dim)
        self._dim = embed_dim
        nn.init.normal_(self.emb.weight, std=0.1)
        nn.init.xavier_uniform_(self.proj.weight)

    @torch.no_grad()
    def encode(self, texts: List[str]) -> torch.Tensor:
        results = []
        for text in texts:
            ids = torch.tensor([min(ord(c), 127) for c in text])
            emb = self.emb(ids).mean(0)
            emb = torch.tanh(self.proj(emb))
            results.append(emb)
        return torch.stack(results)


class SimpleVectorStore:
    def __init__(self):
        self.embeddings: List[torch.Tensor] = []
        self.documents:  List[str]          = []
        self.metadata:   List[Dict]         = []

    def add(self, texts, embeddings, metadata=None):
        for i, (text, emb) in enumerate(zip(texts, embeddings)):
            self.embeddings.append(emb)
            self.documents.append(text)
            self.metadata.append(metadata[i] if metadata else {})

    def search(self, query_emb, top_k=3):
        if not self.embeddings:
            return []
        store = torch.stack(self.embeddings)
        q     = query_emb / (query_emb.norm() + 1e-8)
        s     = store / (store.norm(dim=-1, keepdim=True) + 1e-8)
        sims  = (s @ q).tolist()
        ranked = sorted(zip(sims, self.documents, self.metadata), reverse=True)[:top_k]
        return [(doc, sim, meta) for sim, doc, meta in ranked]


class AnswerHead(nn.Module):
    def __init__(self, embed_dim: int, num_classes: int = 4):
        super().__init__()
        self.attn_pool = nn.Linear(embed_dim, 1)
        self.mlp = nn.Sequential(
            nn.Linear(embed_dim * 2, embed_dim),
            nn.GELU(),
            nn.Linear(embed_dim, num_classes),
        )

    def forward(self, query_emb, retrieved_embs):
        attn_w = torch.softmax(self.attn_pool(retrieved_embs), dim=1)
        pooled = (attn_w * retrieved_embs).sum(dim=1)
        fused  = torch.cat([query_emb, pooled], dim=-1)
        return self.mlp(fused)


# ─────────────────────────────────────────────────────────────────────────────
# FILTER A: WITH FFT  (Spectral Filter)
# ─────────────────────────────────────────────────────────────────────────────

class SpectralFilter(nn.Module):
    """FFT → learnable complex weights → iFFT"""

    def __init__(self, embed_dim: int, dropout: float = 0.1):
        super().__init__()
        freq_dim = embed_dim // 2 + 1
        self.weight_real = nn.Parameter(torch.ones(freq_dim))
        self.weight_imag = nn.Parameter(torch.zeros(freq_dim))
        self.norm        = nn.LayerNorm(embed_dim)
        self.dropout     = nn.Dropout(dropout)
        self._embed_dim  = embed_dim

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        squeeze = x.dim() == 2
        if squeeze:
            x = x.unsqueeze(1)
        residual = x

        # Step 1: FFT
        x_freq = torch.fft.rfft(x, dim=-1)
        # Step 2: Spectral Filter
        weight = torch.complex(self.weight_real, self.weight_imag)
        x_filt = x_freq * weight
        # Step 3: iFFT
        x_out  = torch.fft.irfft(x_filt, n=self._embed_dim, dim=-1)

        x_out = self.norm(x_out + residual)
        x_out = self.dropout(x_out)
        if squeeze:
            x_out = x_out.squeeze(1)
        return x_out

    def num_params(self):
        return sum(p.numel() for p in self.parameters())


# ─────────────────────────────────────────────────────────────────────────────
# FILTER B: WITHOUT FFT  (Standard MLP Filter)
# ─────────────────────────────────────────────────────────────────────────────

class MLPFilter(nn.Module):
    """
    Standard alternative: Linear → GELU → Linear (no frequency domain).
    Same job as SpectralFilter but operates purely in embedding space.
    Parameter count is kept comparable to SpectralFilter for fairness.
    """

    def __init__(self, embed_dim: int, dropout: float = 0.1):
        super().__init__()
        # Comparable param budget to SpectralFilter
        # SpectralFilter has (freq_dim * 2) real params + LayerNorm
        # We use a small hidden MLP of similar size
        hidden = embed_dim // 2 + 1          # same as freq_dim
        self.filter = nn.Sequential(
            nn.Linear(embed_dim, hidden),
            nn.GELU(),
            nn.Linear(hidden, embed_dim),
        )
        self.norm    = nn.LayerNorm(embed_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        squeeze = x.dim() == 2
        if squeeze:
            x = x.unsqueeze(1)
        residual = x

        # Standard spatial-domain transformation (no FFT)
        x_out = self.filter(x)

        x_out = self.norm(x_out + residual)
        x_out = self.dropout(x_out)
        if squeeze:
            x_out = x_out.squeeze(1)
        return x_out

    def num_params(self):
        return sum(p.numel() for p in self.parameters())


# ─────────────────────────────────────────────────────────────────────────────
# GENERIC RAG PIPELINE  (accepts either filter)
# ─────────────────────────────────────────────────────────────────────────────

class RAGPipeline(nn.Module):
    def __init__(self, embed_dim: int = 64, top_k: int = 3,
                 num_classes: int = 4, filter_type: str = "spectral"):
        super().__init__()
        self.embed_dim = embed_dim
        self.top_k     = top_k
        self.name      = filter_type

        self.embedder      = ToyEmbeddingModel(embed_dim)
        self.vector_store  = SimpleVectorStore()
        self.answer_head   = AnswerHead(embed_dim, num_classes)

        if filter_type == "spectral":
            self.emb_filter = SpectralFilter(embed_dim)
        else:
            self.emb_filter = MLPFilter(embed_dim)

    def index_documents(self, documents, metadata=None):
        embs = self.embedder.encode(documents)
        self.vector_store.add(documents, embs, metadata)

    @torch.no_grad()
    def retrieve(self, query):
        q_emb    = self.embedder.encode([query])[0]
        results  = self.vector_store.search(q_emb, top_k=self.top_k)
        texts    = [r[0] for r in results]
        scores   = [r[1] for r in results]
        ret_embs = torch.stack([self.embedder.encode([t])[0] for t in texts])
        return texts, ret_embs, scores

    def forward(self, queries):
        q_embs_list, ret_embs_list = [], []
        for query in queries:
            q_emb     = self.embedder.encode([query])
            results   = self.vector_store.search(q_emb[0], self.top_k)
            ret_texts = [r[0] for r in results]
            ret_embs  = self.embedder.encode(ret_texts)
            q_embs_list.append(q_emb[0])
            ret_embs_list.append(ret_embs)

        q_embs   = torch.stack(q_embs_list)
        ret_embs = torch.stack(ret_embs_list)

        B, K, D  = ret_embs.shape
        flat     = ret_embs.view(B * K, D)
        filtered = self.emb_filter(flat)
        filtered = filtered.view(B, K, D)

        # Track how much the filter changed the embeddings
        self._last_delta = (filtered - ret_embs.view(B, K, D)).abs().mean().item()

        logits = self.answer_head(q_embs, filtered)
        return logits, filtered


# ─────────────────────────────────────────────────────────────────────────────
# DATA
# ─────────────────────────────────────────────────────────────────────────────

DOCUMENTS = [
    "Photosynthesis is the process by which plants convert sunlight into glucose using carbon dioxide and water.",
    "The mitochondria is often called the powerhouse of the cell because it produces ATP through cellular respiration.",
    "DNA is a double helix molecule that carries genetic information in living organisms.",
    "Neurons transmit electrical signals throughout the nervous system, enabling thought and movement.",
    "Black holes are regions of spacetime where gravity is so strong that nothing can escape, not even light.",
    "The French Revolution began in 1789 and led to the rise of Napoleon Bonaparte.",
    "World War II ended in 1945 with the surrender of Germany and Japan.",
    "The Renaissance was a cultural movement that began in Italy during the 14th century.",
    "The printing press, invented by Gutenberg around 1440, revolutionized the spread of information.",
    "The Industrial Revolution transformed manufacturing and society in 18th and 19th century Britain.",
    "Machine learning models learn patterns from data without being explicitly programmed.",
    "The transformer architecture introduced self-attention mechanisms that revolutionized NLP.",
    "Quantum computers use qubits that can exist in superposition, enabling new computational paradigms.",
    "The internet is a global network of interconnected computers communicating via TCP/IP protocols.",
    "Convolutional neural networks excel at image recognition by learning hierarchical visual features.",
]
METADATA = [{"topic": "science"}]*5 + [{"topic": "history"}]*5 + [{"topic": "technology"}]*5

TRAIN_DATA = [
    ("How do plants make food?",                0),
    ("What powers the cell?",                   0),
    ("Explain DNA structure.",                  0),
    ("How do neurons work?",                    0),
    ("What are black holes?",                   0),
    ("What happened in the French Revolution?", 1),
    ("When did World War II end?",              1),
    ("What was the Renaissance?",               1),
    ("Who invented the printing press?",        1),
    ("What was the Industrial Revolution?",     1),
    ("How does machine learning work?",         2),
    ("What is a transformer in AI?",            2),
    ("How do quantum computers work?",          2),
    ("What is the internet?",                   2),
    ("What are convolutional neural networks?", 2),
    ("Tell me about ancient Rome.",             3),
    ("What is the capital of France?",          3),
    ("Who wrote Romeo and Juliet?",             3),
]

TEST_DATA = [
    ("How do plants produce energy from the sun?",    0),
    ("Explain the role of DNA in living cells.",      0),
    ("What caused the fall of Napoleon?",             1),
    ("Describe the printing revolution in history.",  1),
    ("How does a neural network learn?",              2),
    ("What makes quantum computing different?",       2),
    ("What is the tallest mountain?",                 3),
    ("Who painted the Mona Lisa?",                    3),
]

CLASS_NAMES = ["Science", "History", "Technology", "Other"]


# ─────────────────────────────────────────────────────────────────────────────
# TRAINING + EVALUATION
# ─────────────────────────────────────────────────────────────────────────────

def train_pipeline(pipeline: RAGPipeline, epochs: int = 60
                   ) -> Dict:
    trainable = list(pipeline.emb_filter.parameters()) + \
                list(pipeline.answer_head.parameters())
    optimizer = optim.Adam(trainable, lr=3e-3)
    criterion = nn.CrossEntropyLoss()

    queries = [q for q, _ in TRAIN_DATA]
    labels  = torch.tensor([l for _, l in TRAIN_DATA])

    history = {"loss": [], "accuracy": [], "delta": []}

    for epoch in range(1, epochs + 1):
        pipeline.train()
        logits, _ = pipeline(queries)
        loss      = criterion(logits, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        preds   = logits.argmax(-1)
        correct = (preds == labels).sum().item()
        acc     = correct / len(labels)

        history["loss"].append(loss.item())
        history["accuracy"].append(acc)
        history["delta"].append(pipeline._last_delta)

    return history


def evaluate_pipeline(pipeline: RAGPipeline) -> Dict:
    pipeline.eval()
    criterion = nn.CrossEntropyLoss()
    queries   = [q for q, _ in TEST_DATA]
    labels    = torch.tensor([l for _, l in TEST_DATA])

    with torch.no_grad():
        logits, filtered = pipeline(queries)
        loss    = criterion(logits, labels).item()
        preds   = logits.argmax(-1)
        correct = (preds == labels).sum().item()
        acc     = correct / len(labels)
        probs   = torch.softmax(logits, dim=-1)
        conf    = probs.max(dim=-1).values.mean().item()

    return {
        "test_loss":    loss,
        "test_accuracy": acc,
        "avg_confidence": conf,
        "predictions":  preds.tolist(),
        "labels":       labels.tolist(),
    }


# ─────────────────────────────────────────────────────────────────────────────
# PRINT HELPERS
# ─────────────────────────────────────────────────────────────────────────────

def bar(value, max_val=1.0, width=30, char="█"):
    filled = int(value / max_val * width)
    return char * filled + "░" * (width - filled)

def print_header(title):
    print(f"\n{'═'*65}")
    print(f"  {title}")
    print(f"{'═'*65}")

def print_row(label, val_a, val_b, fmt=".4f", higher_better=True):
    a = float(val_a)
    b = float(val_b)
    if higher_better:
        winner = "← FFT wins" if a > b else ("← MLP wins" if b > a else "  tie")
    else:
        winner = "← FFT wins" if a < b else ("← MLP wins" if b < a else "  tie")
    print(f"  {label:<30} {a:{fmt}}   {b:{fmt}}   {winner}")


# ─────────────────────────────────────────────────────────────────────────────
# MAIN
# ─────────────────────────────────────────────────────────────────────────────

def main():
    EPOCHS    = 60
    EMBED_DIM = 64

    print("\n" + "█"*65)
    print("  SPECTRAL RAG (with FFT) vs STANDARD RAG (without FFT)")
    print("  Full Comparison & Evaluation")
    print("█"*65)

    # ── Build both pipelines with identical embedder weights ─────────────────
    spectral_pipe = RAGPipeline(EMBED_DIM, top_k=3, num_classes=4, filter_type="spectral")
    mlp_pipe      = RAGPipeline(EMBED_DIM, top_k=3, num_classes=4, filter_type="mlp")

    # Share the same embedder weights so retrieval is identical
    mlp_pipe.embedder.load_state_dict(spectral_pipe.embedder.state_dict())

    print_header("PARAMETER COUNT")
    sf_params  = spectral_pipe.emb_filter.num_params()
    mlp_params = mlp_pipe.emb_filter.num_params()
    print(f"\n  SpectralFilter (with FFT) : {sf_params:,} trainable params")
    print(f"  MLPFilter (without FFT)   : {mlp_params:,} trainable params")
    print(f"\n  Note: SpectralFilter is more parameter-efficient by design.")
    print(f"  It learns {sf_params} weights to control {EMBED_DIM//2+1} frequency bins.")
    print(f"  MLP learns dense weight matrices ({mlp_params} weights).")

    # ── Index documents (same for both) ──────────────────────────────────────
    print_header("INDEXING")
    spectral_pipe.index_documents(DOCUMENTS, METADATA)
    mlp_pipe.index_documents(DOCUMENTS, METADATA)
    print(f"\n  Both pipelines indexed {len(DOCUMENTS)} documents.")
    print(f"  Training on {len(TRAIN_DATA)} Q&A pairs, testing on {len(TEST_DATA)}.")

    # ── Training ─────────────────────────────────────────────────────────────
    print_header(f"TRAINING  ({EPOCHS} epochs)")
    print(f"\n  {'Epoch':<8} {'FFT Loss':>10} {'FFT Acc':>10} {'MLP Loss':>10} {'MLP Acc':>10}")
    print(f"  {'-'*8} {'-'*10} {'-'*10} {'-'*10} {'-'*10}")

    # Train step by step so we can print jointly
    trainable_s = list(spectral_pipe.emb_filter.parameters()) + \
                  list(spectral_pipe.answer_head.parameters())
    trainable_m = list(mlp_pipe.emb_filter.parameters()) + \
                  list(mlp_pipe.answer_head.parameters())
    opt_s   = optim.Adam(trainable_s, lr=3e-3)
    opt_m   = optim.Adam(trainable_m, lr=3e-3)
    crit    = nn.CrossEntropyLoss()

    queries = [q for q, _ in TRAIN_DATA]
    labels  = torch.tensor([l for _, l in TRAIN_DATA])

    hist_s = {"loss": [], "accuracy": [], "delta": []}
    hist_m = {"loss": [], "accuracy": [], "delta": []}

    for epoch in range(1, EPOCHS + 1):
        # --- spectral ---
        spectral_pipe.train()
        logits_s, _ = spectral_pipe(queries)
        loss_s      = crit(logits_s, labels)
        opt_s.zero_grad(); loss_s.backward(); opt_s.step()
        acc_s = (logits_s.argmax(-1) == labels).float().mean().item()
        hist_s["loss"].append(loss_s.item())
        hist_s["accuracy"].append(acc_s)
        hist_s["delta"].append(spectral_pipe._last_delta)

        # --- mlp ---
        mlp_pipe.train()
        logits_m, _ = mlp_pipe(queries)
        loss_m      = crit(logits_m, labels)
        opt_m.zero_grad(); loss_m.backward(); opt_m.step()
        acc_m = (logits_m.argmax(-1) == labels).float().mean().item()
        hist_m["loss"].append(loss_m.item())
        hist_m["accuracy"].append(acc_m)
        hist_m["delta"].append(mlp_pipe._last_delta)

        if epoch % 10 == 0 or epoch == 1:
            print(f"  {epoch:<8} {loss_s.item():>10.4f} {acc_s:>10.1%} "
                  f"{loss_m.item():>10.4f} {acc_m:>10.1%}")

    # ── Evaluation ───────────────────────────────────────────────────────────
    print_header("TEST SET EVALUATION")
    res_s = evaluate_pipeline(spectral_pipe)
    res_m = evaluate_pipeline(mlp_pipe)

    print(f"\n  {'Metric':<30} {'FFT (Spectral)':>14}   {'No FFT (MLP)':>12}   Winner")
    print(f"  {'-'*30} {'-'*14}   {'-'*12}   {'-'*14}")
    print_row("Test Loss",         res_s["test_loss"],      res_m["test_loss"],      ".4f", higher_better=False)
    print_row("Test Accuracy",     res_s["test_accuracy"],  res_m["test_accuracy"],  ".4f", higher_better=True)
    print_row("Avg Confidence",    res_s["avg_confidence"], res_m["avg_confidence"], ".4f", higher_better=True)
    print_row("Train Loss (final)",hist_s["loss"][-1],      hist_m["loss"][-1],      ".4f", higher_better=False)
    print_row("Train Acc (final)", hist_s["accuracy"][-1],  hist_m["accuracy"][-1],  ".4f", higher_better=True)

    # ── Per-class breakdown ───────────────────────────────────────────────────
    print_header("PER-CLASS ACCURACY")
    print(f"\n  {'Class':<14} {'FFT correct':>12} {'MLP correct':>12}")
    print(f"  {'-'*14} {'-'*12} {'-'*12}")

    for cls_idx, cls_name in enumerate(CLASS_NAMES):
        idxs  = [i for i, (_, l) in enumerate(TEST_DATA) if l == cls_idx]
        s_ok  = sum(1 for i in idxs if res_s["predictions"][i] == cls_idx)
        m_ok  = sum(1 for i in idxs if res_m["predictions"][i] == cls_idx)
        n     = len(idxs)
        sw    = "✓" if s_ok >= m_ok else " "
        mw    = "✓" if m_ok > s_ok  else " "
        print(f"  {cls_name:<14} {s_ok}/{n} ({s_ok/n:.0%}) {sw}     "
              f"{m_ok}/{n} ({m_ok/n:.0%}) {mw}")

    # ── Convergence speed ─────────────────────────────────────────────────────
    print_header("CONVERGENCE SPEED")

    def epoch_first_above(history, threshold):
        for i, v in enumerate(history):
            if v >= threshold:
                return i + 1
        return None

    for thresh in [0.5, 0.7, 0.9]:
        es = epoch_first_above(hist_s["accuracy"], thresh)
        em = epoch_first_above(hist_m["accuracy"], thresh)
        label = f"First epoch ≥{thresh:.0%} train acc"
        es_str = f"epoch {es}" if es else "never"
        em_str = f"epoch {em}" if em else "never"
        winner = ""
        if es and em:
            winner = "← FFT faster" if es < em else ("← MLP faster" if em < es else "tie")
        print(f"  {label:<35} FFT={es_str:<10}  MLP={em_str:<10}  {winner}")

    # ── Embedding delta ───────────────────────────────────────────────────────
    print_header("EMBEDDING TRANSFORMATION ANALYSIS")
    avg_delta_s = np.mean(hist_s["delta"])
    avg_delta_m = np.mean(hist_m["delta"])
    print(f"\n  Mean absolute change to embeddings (training avg):")
    print(f"  SpectralFilter (FFT) : {avg_delta_s:.6f}  {bar(avg_delta_s, max(avg_delta_s, avg_delta_m))}")
    print(f"  MLPFilter (no FFT)   : {avg_delta_m:.6f}  {bar(avg_delta_m, max(avg_delta_s, avg_delta_m))}")

    final_delta_s = hist_s["delta"][-1]
    final_delta_m = hist_m["delta"][-1]
    print(f"\n  Final epoch embedding delta:")
    print(f"  SpectralFilter (FFT) : {final_delta_s:.6f}")
    print(f"  MLPFilter (no FFT)   : {final_delta_m:.6f}")
    print(f"\n  Interpretation: {'FFT' if avg_delta_s > avg_delta_m else 'MLP'} filter "
          f"modifies embeddings more aggressively on average.")

    # ── Frequency weight inspection ───────────────────────────────────────────
    print_header("SPECTRAL FILTER: LEARNED FREQUENCY WEIGHTS")
    wr  = spectral_pipe.emb_filter.weight_real.detach().numpy()
    wi  = spectral_pipe.emb_filter.weight_imag.detach().numpy()
    mag = np.sqrt(wr**2 + wi**2)
    print(f"\n  Frequency bin magnitudes after training (first 16 of {len(mag)}):")
    print(f"  {'Bin':<6} {'Magnitude':>10}  {'Bar':}")
    for i, m in enumerate(mag[:16]):
        print(f"  {i:<6} {m:>10.4f}  {bar(m, mag.max(), width=20)}")
    print(f"\n  → Low-freq bins (0–3): avg magnitude = {mag[:4].mean():.4f}")
    print(f"  → High-freq bins (end): avg magnitude = {mag[-4:].mean():.4f}")
    dom = "Low" if mag[:4].mean() > mag[-4:].mean() else "High"
    print(f"  → Model learned to emphasize {dom}-frequency components.")

    # ── Summary ───────────────────────────────────────────────────────────────
    print_header("SUMMARY & KEY DIFFERENCES")
    print("""
  ┌─────────────────────────┬──────────────────────┬──────────────────────┐
  │ Property                │ With FFT (Spectral)  │ Without FFT (MLP)    │
  ├─────────────────────────┼──────────────────────┼──────────────────────┤
  │ Domain of operation     │ Frequency domain     │ Embedding space      │
  │ What it learns          │ Which freqs matter   │ Linear combinations  │
  │ Inductive bias          │ Global patterns      │ Local correlations   │
  │ Interpretability        │ High (freq analysis) │ Low (black box)      │
  │ Parameter efficiency    │ O(d/2) params        │ O(d²) params         │
  │ Noise filtering         │ Natural (low-pass)   │ Implicit only        │
  │ Suitable for            │ Smooth/global signal │ Local feature mixing │
  └─────────────────────────┴──────────────────────┴──────────────────────┘

  WHY FFT HELPS IN RAG:
  • Retrieved embeddings often carry noise from irrelevant doc parts.
  • FFT exposes global structure — spectral weights can suppress noisy
    high-frequency components and amplify meaningful low-freq signals.
  • The filter is O(d log d) computationally vs O(d²) for an MLP.
  • Learned frequency weights are directly interpretable.

  WHY MLP MIGHT SOMETIMES WIN:
  • More expressive for non-smooth feature interactions.
  • More parameters = more capacity on complex tasks.
  • No assumption that "global frequency structure" matters.
""")
    print("█"*65)
    print("  Done!")
    print("█"*65 + "\n")


if __name__ == "__main__":
    main()


█████████████████████████████████████████████████████████████████
  SPECTRAL RAG (with FFT) vs STANDARD RAG (without FFT)
  Full Comparison & Evaluation
█████████████████████████████████████████████████████████████████

═════════════════════════════════════════════════════════════════
  PARAMETER COUNT
═════════════════════════════════════════════════════════════════

  SpectralFilter (with FFT) : 194 trainable params
  MLPFilter (without FFT)   : 4,449 trainable params

  Note: SpectralFilter is more parameter-efficient by design.
  It learns 194 weights to control 33 frequency bins.
  MLP learns dense weight matrices (4449 weights).

═════════════════════════════════════════════════════════════════
  INDEXING
═════════════════════════════════════════════════════════════════

  Both pipelines indexed 15 documents.
  Training on 18 Q&A pairs, testing on 8.

═════════════════════════════════════════════════════════════════
  TRAINING  (60 epochs)
═══════════════════════════════════════

In [1]:
"""
Spectral RAG Pipeline
=====================
A full Retrieval-Augmented Generation pipeline with FFT-based spectral filtering.

Pipeline:
  Document Chunks
       ↓
  Embedding Model  (e.g. sentence-transformers)
       ↓
  FAISS Vector Store  ← query embedding at inference time
       ↓
  Retrieved Embeddings
       ↓
  [FFT → Spectral Filter (learnable) → iFFT]
       ↓
  Filtered Embeddings → LLM / downstream head

Demo runs fully on CPU with no API keys needed.
"""

import math
import time
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
from typing import List, Tuple, Dict

# ─────────────────────────────────────────────────────────────────────────────
# 1.  SPECTRAL FILTER  (the core novelty)
# ─────────────────────────────────────────────────────────────────────────────

class SpectralFilter(nn.Module):
    """
    Applies FFT → learnable complex weight → iFFT to a batch of embeddings.

    Input:  (batch, seq_len, embed_dim)   or   (batch, embed_dim)
    Output: same shape as input
    """

    def __init__(self, embed_dim: int, dropout: float = 0.1):
        super().__init__()
        freq_dim = embed_dim // 2 + 1          # rfft output size

        # Learnable complex weights (one per frequency bin)
        self.weight_real = nn.Parameter(torch.ones(freq_dim))
        self.weight_imag = nn.Parameter(torch.zeros(freq_dim))

        # Optional: a small feed-forward after the filter for expressiveness
        self.norm    = nn.LayerNorm(embed_dim)
        self.dropout = nn.Dropout(dropout)
        self._embed_dim = embed_dim

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        squeeze = x.dim() == 2          # handle (batch, dim) input
        if squeeze:
            x = x.unsqueeze(1)          # → (batch, 1, dim)

        residual = x

        # ── Step 1: FFT ──────────────────────────────────────────────────────
        x_freq = torch.fft.rfft(x, dim=-1)     # complex, shape (..., freq_dim)

        # ── Step 2: Spectral Filter ──────────────────────────────────────────
        weight  = torch.complex(self.weight_real, self.weight_imag)
        x_filt  = x_freq * weight               # element-wise complex multiply

        # ── Step 3: iFFT ─────────────────────────────────────────────────────
        x_out   = torch.fft.irfft(x_filt, n=self._embed_dim, dim=-1)

        # Residual + norm
        x_out   = self.norm(x_out + residual)
        x_out   = self.dropout(x_out)

        if squeeze:
            x_out = x_out.squeeze(1)
        return x_out


# ─────────────────────────────────────────────────────────────────────────────
# 2.  SIMPLE IN-MEMORY VECTOR STORE  (stands in for FAISS / Chroma / Pinecone)
# ─────────────────────────────────────────────────────────────────────────────

class SimpleVectorStore:
    """Cosine-similarity retrieval over stored embeddings."""

    def __init__(self):
        self.embeddings: List[torch.Tensor] = []
        self.documents:  List[str]          = []
        self.metadata:   List[Dict]         = []

    def add(self, texts: List[str], embeddings: torch.Tensor,
            metadata: List[Dict] = None):
        for i, (text, emb) in enumerate(zip(texts, embeddings)):
            self.embeddings.append(emb)
            self.documents.append(text)
            self.metadata.append(metadata[i] if metadata else {})

    def search(self, query_emb: torch.Tensor, top_k: int = 3
               ) -> List[Tuple[str, float, Dict]]:
        if not self.embeddings:
            return []

        store = torch.stack(self.embeddings)            # (N, dim)
        q     = query_emb / (query_emb.norm() + 1e-8)
        s     = store / (store.norm(dim=-1, keepdim=True) + 1e-8)
        sims  = (s @ q).tolist()

        ranked = sorted(zip(sims, self.documents, self.metadata),
                        reverse=True)[:top_k]
        return [(doc, sim, meta) for sim, doc, meta in ranked]


# ─────────────────────────────────────────────────────────────────────────────
# 3.  TINY EMBEDDING MODEL  (replace with sentence-transformers in production)
# ─────────────────────────────────────────────────────────────────────────────

class ToyEmbeddingModel(nn.Module):
    """
    Deterministic bag-of-characters embedding for demo purposes.
    In a real project swap this for:
        from sentence_transformers import SentenceTransformer
        model = SentenceTransformer('all-MiniLM-L6-v2')
    """

    def __init__(self, embed_dim: int = 64, vocab_size: int = 128):
        super().__init__()
        self.emb     = nn.Embedding(vocab_size, embed_dim)
        self.proj    = nn.Linear(embed_dim, embed_dim)
        self._dim    = embed_dim
        # Fixed weights so the demo is reproducible
        nn.init.normal_(self.emb.weight,  std=0.1)
        nn.init.xavier_uniform_(self.proj.weight)

    @torch.no_grad()
    def encode(self, texts: List[str]) -> torch.Tensor:
        results = []
        for text in texts:
            ids  = torch.tensor([min(ord(c), 127) for c in text])
            emb  = self.emb(ids).mean(0)          # mean pool
            emb  = torch.tanh(self.proj(emb))
            results.append(emb)
        return torch.stack(results)


# ─────────────────────────────────────────────────────────────────────────────
# 4.  DOWNSTREAM HEAD  (simulates an LLM or task-specific classifier)
# ─────────────────────────────────────────────────────────────────────────────

class AnswerHead(nn.Module):
    """
    Takes the spectral-filtered retrieved embeddings + query embedding
    and produces a task output (here: a small classification / scoring head).

    In a real paper you would replace this with:
        - A frozen LLM that receives the filtered embeddings as soft prompts
        - A cross-attention layer over the retrieved docs
    """

    def __init__(self, embed_dim: int, num_classes: int = 4):
        super().__init__()
        self.attn_pool = nn.Linear(embed_dim, 1)    # attention over retrieved docs
        self.mlp = nn.Sequential(
            nn.Linear(embed_dim * 2, embed_dim),
            nn.GELU(),
            nn.Linear(embed_dim, num_classes),
        )

    def forward(self, query_emb: torch.Tensor,
                retrieved_embs: torch.Tensor) -> torch.Tensor:
        """
        query_emb:      (batch, embed_dim)
        retrieved_embs: (batch, top_k, embed_dim)
        returns:        (batch, num_classes)
        """
        # Soft attention pooling over retrieved docs
        attn_w  = torch.softmax(self.attn_pool(retrieved_embs), dim=1)  # (B,K,1)
        pooled  = (attn_w * retrieved_embs).sum(dim=1)                  # (B, dim)

        # Concatenate query with retrieved context
        fused   = torch.cat([query_emb, pooled], dim=-1)                # (B, 2*dim)
        return self.mlp(fused)


# ─────────────────────────────────────────────────────────────────────────────
# 5.  FULL SPECTRAL RAG PIPELINE
# ─────────────────────────────────────────────────────────────────────────────

class SpectralRAGPipeline(nn.Module):
    """
    End-to-end trainable RAG pipeline with spectral filtering.

    Flow:
        query text
            → embedding model (frozen or fine-tuned)
            → vector store retrieval
            → [FFT → spectral filter → iFFT]  ← LEARNABLE
            → answer head (downstream LLM / classifier)
    """

    def __init__(self, embed_dim: int = 64, top_k: int = 3,
                 num_classes: int = 4):
        super().__init__()
        self.embed_dim = embed_dim
        self.top_k     = top_k

        # Sub-modules
        self.embedder       = ToyEmbeddingModel(embed_dim)
        self.vector_store   = SimpleVectorStore()
        self.spectral_filter = SpectralFilter(embed_dim)
        self.answer_head    = AnswerHead(embed_dim, num_classes)

    # ── Index building ────────────────────────────────────────────────────────

    def index_documents(self, documents: List[str],
                        metadata: List[Dict] = None):
        """Embed and store documents in the vector store."""
        print(f"  Indexing {len(documents)} documents …")
        embs = self.embedder.encode(documents)
        self.vector_store.add(documents, embs, metadata)
        print(f"  Done. Vector store size: {len(self.vector_store.documents)}")

    # ── Inference (no gradient) ───────────────────────────────────────────────

    @torch.no_grad()
    def retrieve(self, query: str) -> Tuple[List[str], torch.Tensor]:
        """Embed query, retrieve top-k docs, return texts + embeddings."""
        q_emb    = self.embedder.encode([query])[0]
        results  = self.vector_store.search(q_emb, top_k=self.top_k)
        texts    = [r[0] for r in results]
        scores   = [r[1] for r in results]
        ret_embs = torch.stack([self.embedder.encode([t])[0] for t in texts])
        return texts, ret_embs, scores

    # ── Forward pass (with gradient for training) ─────────────────────────────

    def forward(self, queries: List[str]
                ) -> Tuple[torch.Tensor, torch.Tensor]:
        """
        Returns:
            logits:       (batch, num_classes)
            filtered_embs (batch, top_k, embed_dim)  — for inspection
        """
        q_embs_list, ret_embs_list = [], []

        for query in queries:
            q_emb    = self.embedder.encode([query])            # (1, dim)
            results  = self.vector_store.search(q_emb[0], self.top_k)
            ret_texts = [r[0] for r in results]
            ret_embs  = self.embedder.encode(ret_texts)         # (K, dim)

            q_embs_list.append(q_emb[0])
            ret_embs_list.append(ret_embs)

        q_embs   = torch.stack(q_embs_list)                    # (B, dim)
        ret_embs = torch.stack(ret_embs_list)                  # (B, K, dim)

        # ── Spectral filtering of retrieved embeddings ────────────────────────
        B, K, D    = ret_embs.shape
        flat       = ret_embs.view(B * K, D)
        filtered   = self.spectral_filter(flat)                # (B*K, dim)
        filtered   = filtered.view(B, K, D)

        # ── Answer head ───────────────────────────────────────────────────────
        logits = self.answer_head(q_embs, filtered)            # (B, num_classes)
        return logits, filtered


# ─────────────────────────────────────────────────────────────────────────────
# 6.  DEMO
# ─────────────────────────────────────────────────────────────────────────────

DOCUMENTS = [
    # Science
    "Photosynthesis is the process by which plants convert sunlight into glucose using carbon dioxide and water.",
    "The mitochondria is often called the powerhouse of the cell because it produces ATP through cellular respiration.",
    "DNA is a double helix molecule that carries genetic information in living organisms.",
    "Neurons transmit electrical signals throughout the nervous system, enabling thought and movement.",
    "Black holes are regions of spacetime where gravity is so strong that nothing can escape, not even light.",

    # History
    "The French Revolution began in 1789 and led to the rise of Napoleon Bonaparte.",
    "World War II ended in 1945 with the surrender of Germany and Japan.",
    "The Renaissance was a cultural movement that began in Italy during the 14th century.",
    "The printing press, invented by Gutenberg around 1440, revolutionized the spread of information.",
    "The Industrial Revolution transformed manufacturing and society in 18th and 19th century Britain.",

    # Technology
    "Machine learning models learn patterns from data without being explicitly programmed.",
    "The transformer architecture introduced self-attention mechanisms that revolutionized NLP.",
    "Quantum computers use qubits that can exist in superposition, enabling new computational paradigms.",
    "The internet is a global network of interconnected computers communicating via TCP/IP protocols.",
    "Convolutional neural networks excel at image recognition by learning hierarchical visual features.",
]

METADATA = [{"topic": "science"}] * 5 + \
           [{"topic": "history"}] * 5 + \
           [{"topic": "technology"}] * 5

# Toy Q&A pairs for training demo (query → label index)
TRAIN_DATA = [
    ("How do plants make food?",               0),   # science
    ("What powers the cell?",                  0),
    ("What happened in the French Revolution?",1),   # history
    ("When did World War II end?",             1),
    ("How does machine learning work?",        2),   # technology
    ("What is a transformer in AI?",           2),
    ("Tell me about ancient Rome.",            3),   # out-of-distribution → class 3
    ("What is the capital of France?",         3),
]


def print_section(title: str):
    print(f"\n{'═'*60}")
    print(f"  {title}")
    print(f"{'═'*60}")


def demo_retrieval(pipeline: SpectralRAGPipeline, queries: List[str]):
    print_section("RETRIEVAL DEMO")
    for query in queries:
        print(f"\n  Query: \"{query}\"")
        texts, embs, scores = pipeline.retrieve(query)
        print("  Top retrieved documents:")
        for i, (text, score) in enumerate(zip(texts, scores), 1):
            print(f"    {i}. [sim={score:.3f}] {text[:80]}…"
                  if len(text) > 80 else f"    {i}. [sim={score:.3f}] {text}")


def demo_spectral_filter(pipeline: SpectralRAGPipeline, query: str):
    print_section("SPECTRAL FILTER INSPECTION")
    texts, embs, scores = pipeline.retrieve(query)

    print(f"\n  Query: \"{query}\"")
    print(f"  Retrieved {len(texts)} docs. Applying spectral filter …\n")

    with torch.no_grad():
        # Show one embedding before/after filtering
        flat     = embs                             # (K, dim)
        filtered = pipeline.spectral_filter(flat)

        diff     = (filtered - embs).abs().mean().item()
        print(f"  Embedding dim        : {embs.shape[-1]}")
        print(f"  Num freq components  : {embs.shape[-1] // 2 + 1}")
        print(f"  Mean absolute change : {diff:.6f}")

        # Show the learned frequency weights
        wr = pipeline.spectral_filter.weight_real.detach().numpy()
        wi = pipeline.spectral_filter.weight_imag.detach().numpy()
        mag = np.sqrt(wr**2 + wi**2)
        print(f"\n  Frequency weight magnitudes (first 10 of {len(mag)}):")
        print("  " + "  ".join(f"{m:.3f}" for m in mag[:10]))


def demo_training(pipeline: SpectralRAGPipeline):
    print_section("TRAINING DEMO  (spectral filter learns from Q&A pairs)")

    # Only train the spectral filter + answer head; embedder stays frozen
    trainable = list(pipeline.spectral_filter.parameters()) + \
                list(pipeline.answer_head.parameters())
    optimizer = optim.Adam(trainable, lr=3e-3)
    criterion = nn.CrossEntropyLoss()

    print(f"\n  Trainable parameters: "
          f"{sum(p.numel() for p in trainable):,}")
    print()

    losses = []
    for epoch in range(1, 31):
        epoch_loss = 0.0
        # Mini-batch: all training samples
        queries = [q for q, _ in TRAIN_DATA]
        labels  = torch.tensor([l for _, l in TRAIN_DATA])

        logits, _ = pipeline(queries)
        loss      = criterion(logits, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        epoch_loss = loss.item()
        losses.append(epoch_loss)

        if epoch % 5 == 0:
            preds   = logits.argmax(-1)
            correct = (preds == labels).sum().item()
            print(f"  Epoch {epoch:3d} | loss={epoch_loss:.4f} "
                  f"| accuracy={correct}/{len(labels)}")

    print(f"\n  Training complete. Loss: {losses[0]:.4f} → {losses[-1]:.4f}")


def demo_inference(pipeline: SpectralRAGPipeline):
    print_section("INFERENCE DEMO  (after training)")

    test_queries = [
        "How do plants produce energy from the sun?",
        "Explain the role of DNA in living cells.",
        "What caused World War II to end?",
        "How does a neural network learn?",
    ]

    class_names = ["Science", "History", "Technology", "Other"]

    print()
    with torch.no_grad():
        for query in test_queries:
            logits, filtered_embs = pipeline([query])
            probs  = torch.softmax(logits, dim=-1)[0]
            pred   = probs.argmax().item()
            conf   = probs[pred].item()

            texts, _, scores = pipeline.retrieve(query)

            print(f"  Q: {query}")
            print(f"     Predicted class : {class_names[pred]} "
                  f"(confidence={conf:.1%})")
            print(f"     Top doc         : {texts[0][:70]}…"
                  if len(texts[0]) > 70 else
                  f"     Top doc         : {texts[0]}")
            print(f"     Filtered emb norm: "
                  f"{filtered_embs[0].norm(dim=-1).mean():.4f}")
            print()


def main():
    print("\n" + "█"*60)
    print("  SPECTRAL RAG PIPELINE — Full Demo")
    print("█"*60)

    torch.manual_seed(42)

    # ── Build pipeline ────────────────────────────────────────────────────────
    print_section("SETUP")
    embed_dim = 64
    pipeline  = SpectralRAGPipeline(embed_dim=embed_dim, top_k=3, num_classes=4)

    total_params = sum(p.numel() for p in pipeline.parameters())
    print(f"\n  Embed dim      : {embed_dim}")
    print(f"  Top-K retrieval: {pipeline.top_k}")
    print(f"  Total params   : {total_params:,}")

    # ── Index documents ───────────────────────────────────────────────────────
    print_section("INDEXING")
    pipeline.index_documents(DOCUMENTS, METADATA)

    # ── Retrieval ─────────────────────────────────────────────────────────────
    demo_retrieval(pipeline, [
        "How do plants convert sunlight?",
        "What is machine learning?",
        "Tell me about the French Revolution.",
    ])

    # ── Spectral filter inspection (before training) ──────────────────────────
    demo_spectral_filter(pipeline, "How does a cell produce energy?")

    # ── Training ──────────────────────────────────────────────────────────────
    demo_training(pipeline)

    # ── Spectral filter inspection (after training) ───────────────────────────
    demo_spectral_filter(pipeline, "How does a cell produce energy?")

    # ── Inference ─────────────────────────────────────────────────────────────
    demo_inference(pipeline)

    print("█"*60)
    print("  Done!")
    print("█"*60 + "\n")


if __name__ == "__main__":
    main()


████████████████████████████████████████████████████████████
  SPECTRAL RAG PIPELINE — Full Demo
████████████████████████████████████████████████████████████

════════════════════════════════════════════════════════════
  SETUP
════════════════════════════════════════════════════════════

  Embed dim      : 64
  Top-K retrieval: 3
  Total params   : 21,127

════════════════════════════════════════════════════════════
  INDEXING
════════════════════════════════════════════════════════════
  Indexing 15 documents …
  Done. Vector store size: 15

════════════════════════════════════════════════════════════
  RETRIEVAL DEMO
════════════════════════════════════════════════════════════

  Query: "How do plants convert sunlight?"
  Top retrieved documents:
    1. [sim=0.992] Photosynthesis is the process by which plants convert sunlight into glucose usin…
    2. [sim=0.987] Neurons transmit electrical signals throughout the nervous system, enabling thou…
    3. [sim=0.987] Black holes are re